In [ ]:
"""运行入口"""

import torch
import torch.multiprocessing as mp
import numpy as np
import pandas as pd

import kaiwu as kw
from trainer import Trainer
from saver import Saver
from data_loader import CSVDataset


def get_data_arr(file_path, num_rows=None):
    """
    读取数组并随机采样指定行数（可选）

    参数:
        file_path: 数据文件路径（.npy格式）
        num_rows: 要采样的行数，为None时返回全部数据
    返回:
        处理后的数组（转置后），若采样则返回采样后的结果
    """
    # 加载数据
    data = np.load(file_path)
    print(f"原始数据形状: {data.shape}")

    # 随机采样指定行数（如果num_rows有效）
    if num_rows is not None and num_rows > 0 and num_rows < data.shape[0]:
        # 生成随机索引
        random_indices = np.random.choice(data.shape[0], size=num_rows, replace=False)
        # 根据索引采样
        sampled_data = data[random_indices]
        print(f"采样后数据形状（{num_rows}行）: {sampled_data.shape}")
    else:
        # 不采样，使用全部数据
        sampled_data = data
        if num_rows is not None:
            print("未进行采样（num_rows无效或超出数据范围）")
    return sampled_data.T


def main():
    """自拟数据集测试"""
    num_output = 0  # num_output
    num_visible = 128
    numrow = 100
    max_steps = 101
    data_name = "z_128_100.npy"
    num_hidden = 32
    learning_rate = 0.02
    num_process = 10
    sampler_type = "cim"

    np.random.seed(10)
    data = get_data_arr(f"./data_input/{data_name}")
    data = data.T
    print("data shape", data.shape)

    if sampler_type == "cim":
        kw.common.CheckpointManager.save_dir = './tmp'
        sampler = kw.cim.CIMOptimizer(task_name="test_kpp", wait=True)
        sampler = kw.cim.PrecisionReducer(
            sampler,
            precision=8,
            truncated_precision=10,
            target_bits=550,
            only_feasible_solution=False,
        )
    elif sampler_type == "sa":
        sampler = kw.classical.SimulatedAnnealingOptimizer(
            initial_temperature=100,
            alpha=0.9,
            cutoff_temperature=1e-2,
            iterations_per_t=200,
            size_limit=10,
            rand_seed=1234,
        )
    else:
        raise ValueError(f"Unknown sampler type: {sampler_type}")

    data = CSVDataset(data)
    dataloader = torch.utils.data.DataLoader(data, batch_size=15, shuffle=True)
    saver = Saver()
    trainer = Trainer(
        dataloader,
        saver,
        sampler,
        num_visible=num_visible,
        num_hidden=num_hidden,
        num_output=num_output,
    )

    learning_rate = learning_rate
    weight_decay_rate = 0.01
    momentum_rate = 0.00
    trainer.set_learning_parameters(learning_rate, weight_decay_rate, momentum_rate)
    trainer.set_cost_parameter(alpha=0.5, beta=10)
    save_path = "./test"
    trainer.train(max_steps=max_steps, save_path=save_path, num_processes=num_process)


if __name__ == "__main__":
    kw.common.set_log_level("ERROR")
    # torch并行不支持fork
    mp.set_start_method("spawn", force=True)
    # 强制所有操作在CPU上运行
    torch.cuda.is_available = lambda: False
    main()